In [1]:
import pandas as pd

STRUCTURE_COLS = [
    "hepatic_vein",
    "liver_ligament",
    "black_background",
    "l_hook_electrocautery",
    "grasper",
    "fat",
    "abdominal_wall",
    "gi_tract",
    "blood",
    "connective_tissue",
    "liver",
    "gallbladder",
    "cystic_duct_yellow",
    "cystic_duct_white",
]

TOOL_COLS = ["Grasper", "Bipolar", "Hook", "Scissors", "Clipper", "Irrigator", "SpecimenBag"]


def build_caption_prompt_from_row(row):
    """
    cholec80_full_dataset.csv の 1行 (pd.Series) から
    英語の構造化キャプション(JSON)を生成させるための user プロンプトを作る。
    """
    video = int(row["video"])
    frame = int(row["frame"])
    phase = row.get("phase", "Unknown")

    # segmentationから「存在するクラス名」だけ抽出（列名ベース）
    present_structures = [
        col for col in STRUCTURE_COLS
        if col in row and row[col] > 0
    ]

    # tool annotation から「1のツール」だけ抽出
    present_tools = [
        col for col in TOOL_COLS
        if col in row and row[col] == 1
    ]

    def list2text(lst):
        return ", ".join(lst) if lst else "none"

    structures_text = list2text(present_structures)
    tools_text = list2text(present_tools)

    prompt = f"""You are an expert in describing laparoscopic cholecystectomy scenes.

Below is metadata for a single video frame from the Cholec80 dataset. 
Use only this metadata to infer what is likely visible in the frame and generate a **structured English description**.

[Metadata]
- Dataset: Cholec80
- Video ID: video{video:02d}
- Frame index: {frame}
- Surgical phase label: {phase}
- Segmentation classes present in this frame (column names from the dataset):
  {structures_text}
- Binary tool annotations for this frame (columns with value 1 are present):
  {tools_text}

[Important notes]
- You do NOT see the image itself; you only see the metadata above.
- The segmentation class names are technical feature names from the dataset.
- You should convert them into natural anatomical or surgical terms when appropriate.
  For example, “gallbladder”, “liver”, “hepatic_vein”, “cystic duct”, “abdominal wall”, 
  “gastrointestinal tract”, “blood”, etc.
- For tools, use realistic instrument names such as “grasper”, “bipolar forceps”, “hook cautery”, 
  “scissors”, “clip applier”, “irrigator/suction”, “specimen retrieval bag”, etc.
- Use the surgical phase label to infer what is likely happening (e.g., preparation, dissection, clipping, etc.),
  but do not invent impossible details.

[Output format]
Return ONLY a valid JSON object, with no extra text, no explanation and no markdown.
Use exactly the following keys:

{{
  "caption": string,                 // 1–2 natural English sentences summarizing what is visible in this frame
  "phase_description": string,       // a short phrase describing the surgical phase context in plain English
  "anatomy_present": [string],       // list of anatomical structures or regions likely visible
  "tools_present": [string],         // list of surgical tools/instruments likely visible
  "focus_of_frame": string           // a concise description of what the surgeon is mainly doing or focusing on
}}

Requirements:
- The JSON must be syntactically valid.
- The values should be consistent with the metadata.
- If no tools are present, use an empty list [] for "tools_present".
- If you are unsure about some details, make a reasonable, conservative guess based on the metadata.

Now output ONLY the JSON object for this frame."""

    return prompt


In [3]:
from openai import OpenAI
from pathlib import Path
import pandas as pd
import json
import os

# API key
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Set OPENAI_API_KEY in environment.")
client = OpenAI(api_key=api_key)

df = pd.read_csv("../data/cholec80_sampled_dataset.csv")

# 例：最初の1行だけ
row = df.iloc[0]
prompt = build_caption_prompt_from_row(row)

resp = client.chat.completions.create(
    model="gpt-5-mini-2025-08-07",
    messages=[
        {"role": "system", "content": "You are a helpful medical AI assistant."},
        {"role": "user", "content": prompt},
    ],
    #temperature=0.5,
)

raw = resp.choices[0].message.content
cap = json.loads(raw)  # JSONとしてパース可能なはず

print(cap["caption"])
print(cap["anatomy_present"])
print(cap["tools_present"])

Laparoscopic view shows a grasper retracting tissue near the gallbladder with surrounding liver and adipose tissue visible; the cystic duct region and portions of the abdominal wall and bowel are also seen in the field. The background is the dark laparoscopic field typical of minimally invasive surgery.
['gallbladder', 'liver', 'cystic duct', 'abdominal wall', 'fat (adipose tissue)', 'gastrointestinal tract (bowel)']
['grasper']


In [5]:
cap

{'caption': 'Laparoscopic view shows a grasper retracting tissue near the gallbladder with surrounding liver and adipose tissue visible; the cystic duct region and portions of the abdominal wall and bowel are also seen in the field. The background is the dark laparoscopic field typical of minimally invasive surgery.',
 'phase_description': 'Preparation: exposing and retracting the gallbladder to identify anatomy',
 'anatomy_present': ['gallbladder',
  'liver',
  'cystic duct',
  'abdominal wall',
  'fat (adipose tissue)',
  'gastrointestinal tract (bowel)'],
 'tools_present': ['grasper'],
 'focus_of_frame': 'Retracting the gallbladder with a grasper to expose the cystic duct and surrounding fatty tissue for anatomical identification'}